# Group Recommendation Pipeline — Foursquare Global-scale Check-ins **with User Social Networks**

In [1]:
import subprocess, sys

def sh(cmd, check=True):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr, file=sys.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")
    return p

import importlib
for pkg in ("pandas", "numpy", "networkx"):
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg} {m.__version__}")
    except ModuleNotFoundError:
        print(f"{pkg} not installed — installing")
        sh(f"pip install -q {pkg}")

sh("pip install -q gdown")
import gdown
print("gdown", gdown.__version__)

pandas 2.3.3
numpy 2.0.2
networkx 3.6.1
$ pip install -q gdown
gdown 5.2.2


In [2]:
import gdown

# Dataset #5 "Global-scale Check-in Dataset with User Social Networks"
# https://drive.google.com/file/d/1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8/view
FILE_ID  = "1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8"
ZIP_PATH = "/kaggle/working/dataset_WWW2019.zip"

# fuzzy=True + the /uc? form is the most reliable way through Drive's large-file
# virus-scan interstitial. If this still 403s / returns an HTML page, upload the
# zip as a Kaggle Dataset (see notes above) and set ZIP_PATH to the attached path.
gdown.download(
    url=f"https://drive.google.com/uc?id={FILE_ID}",
    output=ZIP_PATH,
    quiet=False,
    fuzzy=True,
)

Downloading...
From (original): https://drive.google.com/uc?id=1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8
From (redirected): https://drive.google.com/uc?id=1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8&confirm=t&uuid=208596e9-fcd2-46ec-9a40-c167993a0e64
To: /kaggle/working/dataset_WWW2019.zip
100%|██████████| 2.68G/2.68G [00:20<00:00, 128MB/s] 


'/kaggle/working/dataset_WWW2019.zip'

In [3]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.infolist()

print("Number of entries:", len(files))
for f in files:
    print(
        f.filename,
        "| compressed:", round(f.compress_size / 1024**2, 2), "MB",
        "| uncompressed:", round(f.file_size / 1024**2, 2), "MB",
    )

Number of entries: 10
dataset_WWW2019/ | compressed: 0.0 MB | uncompressed: 0.0 MB
dataset_WWW2019/raw_Checkins_anonymized.txt | compressed: 1838.1 MB | uncompressed: 5826.98 MB
__MACOSX/dataset_WWW2019/._raw_Checkins_anonymized.txt | compressed: 0.0 MB | uncompressed: 0.0 MB
dataset_WWW2019/raw_POIs.txt | compressed: 249.55 MB | uncompressed: 672.12 MB
__MACOSX/dataset_WWW2019/._raw_POIs.txt | compressed: 0.0 MB | uncompressed: 0.0 MB
dataset_WWW2019/dataset_WWW_friendship_old.txt | compressed: 1.67 MB | uncompressed: 4.92 MB
dataset_WWW2019/dataset_WWW_readme.txt | compressed: 0.0 MB | uncompressed: 0.0 MB
__MACOSX/dataset_WWW2019/._dataset_WWW_readme.txt | compressed: 0.0 MB | uncompressed: 0.0 MB
dataset_WWW2019/dataset_WWW_Checkins_anonymized.txt | compressed: 467.62 MB | uncompressed: 1471.92 MB
dataset_WWW2019/dataset_WWW_friendship_new.txt | compressed: 2.72 MB | uncompressed: 8.34 MB


### Read the readme, then auto-discover the file names

File names inside this zip differ from TIST2015 (they're the `dataset_WWW_*` family). Rather than hard-code
them, the next two cells (a) print any readme so you can confirm the **column layout**, and (b) pattern-match
the entries so the pipeline keeps working even if the exact names shift slightly. **Skim the readme output** —
the check-in and POI column orders below are assumed to match it.

In [4]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    readme_files = [f for f in z.namelist() if "readme" in f.lower()]

print("readme files:", readme_files)
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for filename in readme_files:
        print("\n" + "=" * 80)
        print(filename)
        print("=" * 80)
        print(z.read(filename).decode("utf-8", errors="replace"))

readme files: ['dataset_WWW2019/dataset_WWW_readme.txt', '__MACOSX/dataset_WWW2019/._dataset_WWW_readme.txt']

dataset_WWW2019/dataset_WWW_readme.txt
This dataset includes long-term (about 22 months from Apr. 2012 to Jan. 2014) global-scale check-in data collected from Foursquare, and also two snapshots of user social networks before and after the check-in data collection period (see more details in our paper). 


The check-in dataset contains 22,809,624 checkins by 114,324 users on 3,820,891 venues. The social network data contains 363,704 (old) and 607,333 (new) friendships.

- File dataset_WWW_Checkins_anonymized.txt contains check-ins with 4 columns, which are:
1. User ID (anonymized)
2. Venue ID (Foursquare, more information see below in raw_POIs.txt)
3. UTC time
4. Timezone offset in minutes (The offset in minutes between when this check-sin occurred and the same time in UTC, i.e., UTC time + offset is the local time)

- File dataset_WWW_friendship_old.txt and dataset_WWW_friends

In [5]:
# Auto-discover the entries we care about by pattern. The expected members are:
#   - anonymized check-ins : dataset_WWW_Checkins_anonymized.txt
#   - friendship (old/new) : dataset_WWW_friendship_old.txt / _new.txt
#   - POI coordinates      : raw_POIs.txt
import re

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

def pick(pred, names):
    hits = [n for n in names if pred(n.lower()) and not n.endswith("/")]
    return hits

checkin_entry = pick(lambda n: "checkin" in n and "anonym" in n and "raw" not in n, names)
if not checkin_entry:  # fall back to any non-raw checkins file
    checkin_entry = pick(lambda n: "checkin" in n and "raw" not in n, names)
friend_old_entry = pick(lambda n: "friend" in n and "old" in n, names)
friend_new_entry = pick(lambda n: "friend" in n and "new" in n, names)
poi_entry = pick(lambda n: "poi" in n, names)

print("check-ins    :", checkin_entry)
print("friendship old:", friend_old_entry)
print("friendship new:", friend_new_entry)
print("POIs         :", poi_entry)

assert checkin_entry, "Could not find the anonymized check-ins file — inspect z.namelist() above."
assert poi_entry,     "Could not find a POIs file — inspect z.namelist() above."
assert friend_old_entry or friend_new_entry, "Could not find any friendship file."

CHECKIN_ENTRY    = checkin_entry[0]
POI_ENTRY        = poi_entry[0]
FRIEND_OLD_ENTRY = friend_old_entry[0] if friend_old_entry else None
FRIEND_NEW_ENTRY = friend_new_entry[0] if friend_new_entry else None

check-ins    : ['dataset_WWW2019/dataset_WWW_Checkins_anonymized.txt']
friendship old: ['dataset_WWW2019/dataset_WWW_friendship_old.txt']
friendship new: ['dataset_WWW2019/dataset_WWW_friendship_new.txt']
POIs         : ['dataset_WWW2019/raw_POIs.txt', '__MACOSX/dataset_WWW2019/._raw_POIs.txt']


In [6]:
import os

# Extract only what we need (skip the giant raw_* dump).
WORK = "/kaggle/working"
to_extract = [CHECKIN_ENTRY, POI_ENTRY] + [e for e in (FRIEND_OLD_ENTRY, FRIEND_NEW_ENTRY) if e]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for entry in to_extract:
        print("Extracting", entry, "...")
        z.extract(entry, WORK)

# Resolve to on-disk paths (zip entries may include a top-level folder)
CHECKIN_FILE    = os.path.join(WORK, CHECKIN_ENTRY)
POI_FILE        = os.path.join(WORK, POI_ENTRY)
FRIEND_OLD_FILE = os.path.join(WORK, FRIEND_OLD_ENTRY) if FRIEND_OLD_ENTRY else None
FRIEND_NEW_FILE = os.path.join(WORK, FRIEND_NEW_ENTRY) if FRIEND_NEW_ENTRY else None
print("check-ins :", CHECKIN_FILE)
print("POIs      :", POI_FILE)
print("friend old:", FRIEND_OLD_FILE)
print("friend new:", FRIEND_NEW_FILE)

Extracting dataset_WWW2019/dataset_WWW_Checkins_anonymized.txt ...
Extracting dataset_WWW2019/raw_POIs.txt ...
Extracting dataset_WWW2019/dataset_WWW_friendship_old.txt ...
Extracting dataset_WWW2019/dataset_WWW_friendship_new.txt ...
check-ins : /kaggle/working/dataset_WWW2019/dataset_WWW_Checkins_anonymized.txt
POIs      : /kaggle/working/dataset_WWW2019/raw_POIs.txt
friend old: /kaggle/working/dataset_WWW2019/dataset_WWW_friendship_old.txt
friend new: /kaggle/working/dataset_WWW2019/dataset_WWW_friendship_new.txt


### City filtering via POI coordinates

`raw_POIs.txt` gives `venue_id → (lat, lon, category, country)`. We filter POIs to three city bounding boxes,
then keep only check-ins at those venues.

> **One assumption to confirm at runtime:** that the `venue_id` in the *anonymized check-ins* uses the same
> id space as `raw_POIs.txt`. The diagnostic cell below prints the **join match rate**. If it's high (say
> >90%), city filtering is valid. If it's low, the anonymized and raw id spaces differ — in that case use the
> **global fallback** further down (no coordinates needed) or consult the readme for a venue-id mapping.

In [7]:
import pandas as pd

# POI schema (per TIST2015/WWW readme): venue_id, lat, lon, category, country
pois = pd.read_csv(
    POI_FILE, sep="\t", header=None,
    names=["venue_id", "lat", "lon", "category", "country"],
    dtype={"venue_id": str}, low_memory=False,
)
print("Total POIs:", len(pois))
pois.head()

Total POIs: 11180160


,venue_id,lat,lon,category,country
0,3fd66200f964a52000e61ee3,40.729209,-73.998753,Post Office,US
1,3fd66200f964a52000e71ee3,40.733596,-74.003139,Jazz Club,US
2,3fd66200f964a52000e81ee3,40.758102,-73.975734,Gym,US
3,3fd66200f964a52000ea1ee3,40.732456,-74.003755,Indian Restaurant,US
4,3fd66200f964a52000ec1ee3,42.345907,-71.087001,Indian Restaurant,US


In [8]:
# City bounding boxes (same as the previous notebook)
CITY_BBOX = {
    "New York":    {"lon_min": -74.3,  "lon_max": -73.6,  "lat_min": 40.4, "lat_max": 41.0},
    "Chicago":     {"lon_min": -88.0,  "lon_max": -87.5,  "lat_min": 41.6, "lat_max": 42.1},
    "Los Angeles": {"lon_min": -118.7, "lon_max": -117.6, "lat_min": 33.6, "lat_max": 34.4},
}

city_pois = {}
for city, b in CITY_BBOX.items():
    mask = pois["lon"].between(b["lon_min"], b["lon_max"]) & pois["lat"].between(b["lat_min"], b["lat_max"])
    city_pois[city] = pois.loc[mask].copy()
    print(f"{city}: {len(city_pois[city]):,} POIs")

NYC_VENUES = set(city_pois["New York"]["venue_id"])
CHI_VENUES = set(city_pois["Chicago"]["venue_id"])
LA_VENUES  = set(city_pois["Los Angeles"]["venue_id"])
TARGET_VENUES = NYC_VENUES | CHI_VENUES | LA_VENUES
print("Total target venues:", len(TARGET_VENUES))

New York: 113,326 POIs
Chicago: 41,485 POIs
Los Angeles: 82,917 POIs
Total target venues: 237728


In [9]:
# DIAGNOSTIC: does the anonymized check-in venue_id space actually join to raw_POIs?
# Sample the first chunk of check-ins and measure overlap with the POI id set.
poi_ids = set(pois["venue_id"])
sample = pd.read_csv(
    CHECKIN_FILE, sep="\t", header=None,
    names=["user_id", "venue_id", "utc_time", "timezone_offset"],
    dtype={"user_id": str, "venue_id": str}, nrows=200_000,
)
match_rate = sample["venue_id"].isin(poi_ids).mean()
print(f"venue_id join match rate on first 200k check-ins: {match_rate:.1%}")

venue_id join match rate on first 200k check-ins: 100.0%


In [10]:
# Filter the big check-in file chunk-by-chunk into per-city files.
# (If the diagnostic above showed a low match rate, skip this and use the global fallback instead.)
CHUNK_SIZE = 500_000
output_files = {
    "New York":    "/kaggle/working/checkins_new_york.txt",
    "Chicago":     "/kaggle/working/checkins_chicago.txt",
    "Los Angeles": "/kaggle/working/checkins_los_angeles.txt",
}
venue_sets = {"New York": NYC_VENUES, "Chicago": CHI_VENUES, "Los Angeles": LA_VENUES}

for path in output_files.values():
    if os.path.exists(path):
        os.remove(path)
first_write = {c: True for c in output_files}
total_kept  = {c: 0 for c in output_files}
total_seen  = 0

for chunk in pd.read_csv(
    CHECKIN_FILE, sep="\t", header=None,
    names=["user_id", "venue_id", "utc_time", "timezone_offset"],
    dtype={"user_id": str, "venue_id": str}, chunksize=CHUNK_SIZE,
):
    total_seen += len(chunk)
    for city, vset in venue_sets.items():
        sub = chunk[chunk["venue_id"].isin(vset)]
        if len(sub):
            sub.to_csv(output_files[city], sep="\t", index=False,
                       header=first_write[city], mode="a")
            first_write[city] = False
            total_kept[city] += len(sub)
    print(f"Processed: {total_seen:,} | " +
          " | ".join(f"{c}: {total_kept[c]:,}" for c in output_files))

Processed: 500,000 | New York: 7,945 | Chicago: 3,008 | Los Angeles: 4,387
Processed: 1,000,000 | New York: 16,166 | Chicago: 6,086 | Los Angeles: 8,070
Processed: 1,500,000 | New York: 24,009 | Chicago: 8,748 | Los Angeles: 12,150
Processed: 2,000,000 | New York: 31,679 | Chicago: 11,435 | Los Angeles: 16,052
Processed: 2,500,000 | New York: 39,384 | Chicago: 14,178 | Los Angeles: 19,962
Processed: 3,000,000 | New York: 47,344 | Chicago: 17,057 | Los Angeles: 24,043
Processed: 3,500,000 | New York: 55,211 | Chicago: 20,028 | Los Angeles: 27,834
Processed: 4,000,000 | New York: 62,164 | Chicago: 22,710 | Los Angeles: 31,603
Processed: 4,500,000 | New York: 69,349 | Chicago: 25,598 | Los Angeles: 35,422
Processed: 5,000,000 | New York: 76,221 | Chicago: 28,483 | Los Angeles: 39,244
Processed: 5,500,000 | New York: 83,068 | Chicago: 31,167 | Los Angeles: 42,955
Processed: 6,000,000 | New York: 89,650 | Chicago: 34,184 | Los Angeles: 46,731
Processed: 6,500,000 | New York: 96,436 | Chicag

In [11]:
# Save the per-city POI files alongside (same as the previous notebook)
CITY_OUTPUT_DIRS = {
    "New York":    "/kaggle/working/New_York",
    "Chicago":     "/kaggle/working/Chicago",
    "Los Angeles": "/kaggle/working/Los_Angeles",
}
for city, out_dir in CITY_OUTPUT_DIRS.items():
    os.makedirs(out_dir, exist_ok=True)
    poi_out = os.path.join(out_dir, "pois.txt")
    city_pois[city].to_csv(poi_out, sep="\t", index=False)
    print(f"{city}: saved {len(city_pois[city]):,} POIs -> {poi_out}")

New York: saved 113,326 POIs -> /kaggle/working/New_York/pois.txt
Chicago: saved 41,485 POIs -> /kaggle/working/Chicago/pois.txt
Los Angeles: saved 82,917 POIs -> /kaggle/working/Los_Angeles/pois.txt


## 2. Group construction (CubeRec, Chen et al. SIGIR'22) — using the **real** social graph

We follow Section 4.1 of *"Thinking inside The Box: Learning Hypercube Representations for Group
Recommendation."* A **group** is a set of **socially-connected** users who check in at the **same venue**
within the **same time window**.

**What changed vs. the TIST2015 version:** that dump had no friendship file, so it approximated the social
graph with a co-visitation proxy (`MIN_COOCCURRENCE`). This dataset ships **real friendship edges**, so we
load them directly (`dataset_WWW_friendship_old.txt` ∪ `_new.txt`) and induce connected components on the
genuine graph. The rest of the rule — bucket check-ins by (venue, time-window), then take socially-connected
components within each bucket as groups — is exactly the paper's.

**Two modelling choices worth knowing:**
- *Which friendship snapshot?* The check-ins span Apr 2012–Jan 2014; `friendship_old` predates collection and
  `friendship_new` postdates it. We default to the **union** (a tie in either snapshot counts as "connected").
  Set `FRIENDSHIP = "old"` or `"new"` to restrict.
- *Component vs. clique?* The paper says "a set of users who are connected on the social network." We read that
  as a **connected component** of the induced subgraph (matches the previous notebook). For a stricter reading,
  swap `nx.connected_components` for `nx.find_cliques` — noted inline.

In [12]:
import pandas as pd
import numpy as np
import networkx as nx
import os

# ---------------------------------------------------------------
# Config
# ---------------------------------------------------------------
TIME_WINDOW_MINUTES = 720        # "same time" bucket width
MIN_GROUP_SIZE       = 2         
MAX_GROUP_SIZE       = 20        
MAX_HOPS             = 5
FRIENDSHIP           = "union"   # "old" | "new" | "union"


In [13]:
def load_real_social_graph(friend_old=FRIEND_OLD_FILE, friend_new=FRIEND_NEW_FILE, which=FRIENDSHIP):
    def _read(path):
        e = pd.read_csv(path, sep="\t", header=None, names=["u", "v"],
                        dtype=str, low_memory=False)
        return list(zip(e["u"], e["v"]))

    edges = []
    if which in ("old", "union") and friend_old:
        edges += _read(friend_old)
    if which in ("new", "union") and friend_new:
        edges += _read(friend_new)

    G = nx.Graph()
    G.add_edges_from(edges)
    print(f"Real social graph ({which}): {G.number_of_nodes():,} users, "
          f"{G.number_of_edges():,} friendships")
    return G


def load_city_checkins(path: str) -> pd.DataFrame:
    """Load a per-city checkin file and add a `local_time` column using
    utc_time + timezone_offset (per the readme's definition of local time)."""
    df = pd.read_csv(path, sep="\t", dtype={"user_id": str, "venue_id": str})
    df["utc_time"] = pd.to_datetime(
        df["utc_time"], format="%a %b %d %H:%M:%S %z %Y", errors="coerce"
    )
    df = df.dropna(subset=["utc_time"])
    df["local_time"] = df["utc_time"] + pd.to_timedelta(df["timezone_offset"], unit="m")
    df["local_time"] = df["local_time"].dt.tz_localize(None)
    return df


def venue_time_clusters(checkins: pd.DataFrame,
                        time_window_minutes: int = TIME_WINDOW_MINUTES) -> pd.Series:
    df = checkins.copy()
    df["ts_bucket"] = df["local_time"].dt.floor(f"{time_window_minutes}min")
    clusters = (df.groupby(["venue_id", "ts_bucket"])["user_id"]
                  .apply(lambda s: sorted(set(s))))
    clusters = clusters[clusters.apply(len) >= 2]
    return clusters


def build_groups(clusters: pd.Series, social_graph: nx.Graph,
                 min_group_size: int = MIN_GROUP_SIZE,
                 max_group_size: int = MAX_GROUP_SIZE,
                 max_hops: int = MAX_HOPS) -> pd.DataFrame:
    reachable_cache: dict = {}

    def reachable_within(u):
        if u not in reachable_cache:
            reachable_cache[u] = nx.single_source_shortest_path_length(
                social_graph, u, cutoff=max_hops
            )
        return reachable_cache[u]

    groups, gid = [], 0
    for (venue_id, ts_bucket), users in clusters.items():
        present = [u for u in users if u in social_graph]
        if len(present) < min_group_size:
            continue
        present_set = set(present)
        sub = nx.Graph()
        sub.add_nodes_from(present)
        for u in present:
            for v, dist in reachable_within(u).items():
                if v != u and v in present_set and 0 < dist <= max_hops:
                    sub.add_edge(u, v)
        # --- connected components (paper reading). For a stricter clique reading:
        # for comp in nx.find_cliques(sub):
        for comp in nx.connected_components(sub):
            comp = sorted(comp)
            if min_group_size <= len(comp) <= max_group_size:
                groups.append({"group_id": gid, "venue_id": venue_id,
                               "ts_bucket": ts_bucket, "members": comp})
                gid += 1
    return pd.DataFrame(groups)


def groups_to_cuberec_format(groups: pd.DataFrame, user2idx: dict, venue2idx: dict):
    """Emit gu.dat / gi.dat lines matching CubeRec's datapre.py loader:
    'parent<TAB>child1,child2,...' where parent=group_id."""
    gu_lines, gi_lines = [], []
    for _, row in groups.iterrows():
        gid = row["group_id"]
        member_idxs = [str(user2idx[u]) for u in row["members"]]
        gu_lines.append(f"{gid}\t{','.join(member_idxs)}")
        gi_lines.append(f"{gid}\t{venue2idx[row['venue_id']]}")
    return gu_lines, gi_lines


def social_to_cuberec_format(social_graph: nx.Graph, user2idx: dict):
    """Emit social.dat lines ('user<TAB>friend1,friend2,...') for CubeRec's
    socially-enhanced LightGCN (Eq. 3-4), restricted to users in this city."""
    lines = []
    for u, idx in sorted(user2idx.items(), key=lambda kv: kv[1]):
        if u in social_graph:
            friends = [str(user2idx[f]) for f in social_graph.neighbors(u) if f in user2idx]
            if friends:
                lines.append(f"{idx}\t{','.join(sorted(set(friends), key=int))}")
    return lines

In [14]:
# === LLMGPR §4.1 filtering: remove users and POIs with < 10 interactions ===
MIN_USER_CHECKINS = 1   # drop users with fewer than this many check-ins
MIN_POI_CHECKINS  = 1  # drop POIs  with fewer than this many check-ins

def filter_k_core(checkins, min_user=MIN_USER_CHECKINS, min_poi=MIN_POI_CHECKINS):
    """Iteratively remove low-activity users and POIs (bipartite k-core).
    Follows LLMGPR §4.1: 'users and POIs with less than 10 interactions are
    removed.' Iterates because removing a user can drop a POI below threshold
    and vice-versa, so a single pass is not enough.

    'interactions' is read as check-in count (rows). For the distinct-POI
    reading instead, swap the value_counts lines for .groupby(...).nunique().
    """
    df = checkins
    before = len(df)
    while True:
        uc = df["user_id"].value_counts()
        df = df[df["user_id"].isin(uc[uc > min_user].index)]
        vc = df["venue_id"].value_counts()
        df = df[df["venue_id"].isin(vc[vc > min_poi].index)]
        if len(df) == 0:
            break
        u_ok = (df["user_id"].value_counts()  > min_user).all()
        v_ok = (df["venue_id"].value_counts() > min_poi).all()
        if u_ok and v_ok:
            break
    print(f"  k-core(≥{min_user}u/≥{min_poi}p): {before:,} → {len(df):,} check-ins, "
          f"{df['user_id'].nunique():,} users, {df['venue_id'].nunique():,} POIs")
    return df.copy()

### Run per city

`gu.dat` / `gi.dat` are written in the exact `parent\tchild1,child2,...` format expected by CubeRec's own
`datapre.py` (`load_dat`), so you can drop them straight into `CubeRec/code/data/<city>/` and run their
pipeline unmodified. Unlike the TIST2015 version, we can now also export a **real `social.dat`** (the induced
friendship graph on each city's users) for the socially-enhanced LightGCN in Eq. 3–4 — no empty/identity
placeholder needed.

If you used the global fallback, set `CITY_CHECKIN_FILES = {"Global": "/kaggle/working/checkins_global.txt"}`.

In [15]:
def run_group_pipeline(checkin_path: str, out_dir: str, social_graph: nx.Graph):
    os.makedirs(out_dir, exist_ok=True)

    checkins = load_city_checkins(checkin_path)
    print(f"Loaded {len(checkins):,} checkins, "
          f"{checkins['user_id'].nunique():,} users, "
          f"{checkins['venue_id'].nunique():,} venues")

    # >>> ADDED: LLMGPR §4.1 — remove users/POIs with < 10 interactions <
    checkins = filter_k_core(checkins)

    clusters = venue_time_clusters(checkins)
    print(f"Candidate (venue, time-window) buckets: {len(clusters):,}")

    groups = build_groups(clusters, social_graph)
    print(f"Groups formed: {len(groups):,}")
    if len(groups):
        sizes = groups["members"].apply(len)
        print(f"Avg group size: {sizes.mean():.2f} (min {sizes.min()}, max {sizes.max()})")

    all_users  = sorted(checkins["user_id"].unique())
    all_venues = sorted(checkins["venue_id"].unique())
    user2idx  = {u: i for i, u in enumerate(all_users)}
    venue2idx = {v: i for i, v in enumerate(all_venues)}

    ui = checkins[["user_id", "venue_id"]].drop_duplicates()
    ui_lines = (ui.assign(u=ui["user_id"].map(user2idx), v=ui["venue_id"].map(venue2idx))
                  .groupby("u")["v"].apply(lambda s: ",".join(map(str, sorted(set(s))))))
    with open(os.path.join(out_dir, "ui.dat"), "w") as f:
        for u, items in ui_lines.items():
            f.write(f"{u}\t{items}\n")

    social_lines = social_to_cuberec_format(social_graph, user2idx)
    with open(os.path.join(out_dir, "social.dat"), "w") as f:
        f.write("\n".join(social_lines) + ("\n" if social_lines else ""))

    if len(groups):
        gu_lines, gi_lines = groups_to_cuberec_format(groups, user2idx, venue2idx)
        with open(os.path.join(out_dir, "gu.dat"), "w") as f:
            f.write("\n".join(gu_lines) + "\n")
        with open(os.path.join(out_dir, "gi.dat"), "w") as f:
            f.write("\n".join(gi_lines) + "\n")

    groups.to_pickle(os.path.join(out_dir, "groups.pkl"))
    print(f"Wrote ui.dat, social.dat, gu.dat, gi.dat, groups.pkl -> {out_dir}")
    return checkins, groups

In [16]:
# Load the real social graph ONCE, then reuse it for every city.
SOCIAL_GRAPH = load_real_social_graph()

CITY_CHECKIN_FILES = {
    "New York":    "/kaggle/working/checkins_new_york.txt",
    "Chicago":     "/kaggle/working/checkins_chicago.txt",
    "Los Angeles": "/kaggle/working/checkins_los_angeles.txt",
}
CITY_OUT_DIRS = {
    "New York":    "/kaggle/working/New_York/cuberec_groups",
    "Chicago":     "/kaggle/working/Chicago/cuberec_groups",
    "Los Angeles": "/kaggle/working/Los_Angeles/cuberec_groups",
}

city_results = {}
for city in CITY_CHECKIN_FILES:
    print(f"\n=== {city} ===")
    checkins, groups = run_group_pipeline(
        CITY_CHECKIN_FILES[city], CITY_OUT_DIRS[city], SOCIAL_GRAPH
    )
    city_results[city] = {"checkins": checkins, "groups": groups}

Real social graph (union): 114,324 users, 701,317 friendships

=== New York ===
Loaded 302,187 checkins, 9,141 users, 47,826 venues
  k-core(≥1u/≥1p): 302,187 → 279,259 check-ins, 7,336 users, 26,564 POIs
Candidate (venue, time-window) buckets: 14,807
Groups formed: 11,994
Avg group size: 2.74 (min 2, max 20)
Wrote ui.dat, social.dat, gu.dat, gi.dat, groups.pkl -> /kaggle/working/New_York/cuberec_groups

=== Chicago ===
Loaded 125,538 checkins, 4,568 users, 19,668 venues
  k-core(≥1u/≥1p): 125,538 → 115,749 check-ins, 3,255 users, 11,100 POIs
Candidate (venue, time-window) buckets: 4,736
Groups formed: 4,156
Avg group size: 2.84 (min 2, max 20)
Wrote ui.dat, social.dat, gu.dat, gi.dat, groups.pkl -> /kaggle/working/Chicago/cuberec_groups

=== Los Angeles ===
Loaded 164,616 checkins, 5,515 users, 35,047 venues
  k-core(≥1u/≥1p): 164,616 → 146,628 check-ins, 4,093 users, 18,334 POIs
Candidate (venue, time-window) buckets: 5,701
Groups formed: 4,526
Avg group size: 2.75 (min 2, max 20)
Wr

In [17]:
# Sanity check: reproduce the shape of Table 1 stats from the paper
print(f"{'city':<12} {'#users':>10} {'#items':>10} {'#groups':>10} {'avg_grp_sz':>11} {'grp_int':>10}")
for city, res in city_results.items():
    groups   = res["groups"]
    checkins = res["checkins"]
    n_users  = checkins["user_id"].nunique()
    n_items  = checkins["venue_id"].nunique()
    n_groups = len(groups)
    avg_sz   = groups["members"].apply(len).mean() if n_groups else float("nan")
    # group-item interactions = one per (group, venue) event row
    grp_int  = n_groups  # each group row is tied to exactly one venue event here
    print(f"{city:<12} {n_users:>10,} {n_items:>10,} {n_groups:>10,} {avg_sz:>11.2f} {grp_int:>10,}")

city             #users     #items    #groups  avg_grp_sz    grp_int
New York          7,336     26,564     11,994        2.74     11,994
Chicago           3,255     11,100      4,156        2.84      4,156
Los Angeles       4,093     18,334      4,526        2.75      4,526


In [18]:
# === Dataset statistics table (overall dataset) ===
import pandas as pd
import numpy as np

# venue_id -> category lookup
try:
    venue2cat = dict(zip(pois["venue_id"], pois["category"]))
except NameError:
    venue2cat = {}

ROWS = [
    "#users",
    "#groups",
    "#POIs",
    "#categories",
    "#user check-ins",
    "#group check-ins",
    "#check-ins per user",
    "#check-ins per group",
    "#users per group"
]

# Combine check-ins and groups from all cities
all_checkins = pd.concat(
    [res["checkins"] for res in city_results.values()],
    ignore_index=True
)

all_groups = pd.concat(
    [res["groups"] for res in city_results.values()],
    ignore_index=True
)

# Overall statistics
n_users = all_checkins["user_id"].nunique()
n_user_ckins = len(all_checkins)

n_pois = all_checkins["venue_id"].nunique()

n_categories = (
    all_checkins["venue_id"]
    .map(venue2cat)
    .nunique()
    if venue2cat else float("nan")
)

n_group_ckins = len(all_groups)

if n_group_ckins:
    member_sets = all_groups["members"].apply(lambda m: frozenset(m))
    n_groups = member_sets.nunique()
    users_per_g = all_groups["members"].apply(len).mean()
else:
    n_groups = 0
    users_per_g = float("nan")

table = {
    "#users": n_users,
    "#groups": n_groups,
    "#POIs": n_pois,
    "#categories": n_categories,
    "#user check-ins": n_user_ckins,
    "#group check-ins": n_group_ckins,
    "#check-ins per user": (
        n_user_ckins / n_users if n_users else float("nan")
    ),
    "#check-ins per group": (
        n_group_ckins / n_groups if n_groups else float("nan")
    ),
    "#users per group": users_per_g,
}

stats = pd.DataFrame(
    {"Overall": table}
).reindex(ROWS)

def _fmt(x):
    if pd.isna(x):
        return "—"
    if isinstance(x, (float, np.floating)) and not float(x).is_integer():
        return f"{x:,.2f}"
    return f"{int(x):,}"

stats_display = stats.applymap(_fmt)

print(stats_display.to_string())
stats_display

                      Overall
#users                 11,715
#groups                17,701
#POIs                  55,998
#categories               412
#user check-ins       541,636
#group check-ins       20,676
#check-ins per user     46.23
#check-ins per group     1.17
#users per group         2.76


/tmp/ipykernel_58/2001084539.py:84: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  stats_display = stats.applymap(_fmt)


,Overall
#users,"11,715"
#groups,"17,701"
#POIs,"55,998"
#categories,412
#user check-ins,"541,636"
#group check-ins,"20,676"
#check-ins per user,46.23
#check-ins per group,1.17
#users per group,2.76
